### 1. This section sets up the parameters for the Hidden Markov Model used in the Viterbi algorithm:


In [1]:
states = ['E', '5', 'I']
start_prob = {'E': 1.0, '5': 0.0, 'I': 0.0}

trans_prob = {
    'E': {'E': 0.9, '5': 0.1, 'I': 0.0},
    '5': {'I': 1.0, 'E': 0.0, '5': 0.0},
    'I': {'I': 0.9, 'E': 0.1, '5': 0.0},
}

emit_prob = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'G': 0.95, 'C': 0.0, 'T': 0.0},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}
}

### 2. Calculate Log Probability of a Given Path



In [ ]:
import math

def get_log_prob_of_path(state_path,
                         sequence,
                         start_prob,
                         trans_prob,
                         emit_prob):
    """
    Compute log-probability of emitting `sequence` while following
    `state_path` through an HMM defined by start_prob, trans_prob, emit_prob.
    """
    if len(state_path) != len(sequence):
        raise ValueError("state_path and sequence must have the same length")

    logp = 0.0

    s0 = state_path[0]
    logp += math.log(start_prob[s0])
    logp += math.log(emit_prob[s0][sequence[0]])

    for prev_state, curr_state, sym in zip(
            state_path, state_path[1:], sequence[1:]):
        logp += math.log(trans_prob[prev_state][curr_state])
        logp += math.log(emit_prob[curr_state][sym])

    return round(logp, 2)


state_path = "EEEEEEEEEEEEEEE5IIIIIIIIII"
sequence   = "CTTCATGTGAAAGCAGACGTAAGTCA"

print("Log probability of given path:",
    get_log_prob_of_path(
        state_path,
        sequence,
        start_prob,
        trans_prob,
        emit_prob
    )
)

Log probability of given path: -40.28


### 3. Viterbi Algorithm Implementation

This section contains the implementation of the Viterbi algorithm, which determines the most likely sequence of hidden states that could generate a given observed nucleotide sequence.

Main steps:

- Initialization: Set the starting probabilities for the first observed symbol across all states.

- Recursion: At each subsequent position, calculate the most probable path to each state based on previous probabilities, transition likelihoods, and emission values.

- Termination: Identify the state sequence that has the highest overall probability at the end of the sequence.

In [ ]:
import math

def viterbi(obs_seq, states, start_prob, trans_prob, emit_prob):
    """
    Viterbi algorithm to find the most probable sequence of states given an observation sequence.

    Parameters:
    - obs_seq: list or string of observed symbols (e.g., nucleotides)
    - states: list of possible hidden states
    - start_prob: dict of starting probabilities for each state
    - trans_prob: nested dict of transition probabilities [from][to]
    - emit_prob: nested dict of emission probabilities [state][symbol]

    Returns:
    - most probable state path as a list of state symbols
    """
    V = [{}]  
    path = {} 
    epsilon = 1e-10  


    first_obs = obs_seq[0]
    for state in states:
        V[0][state] = (
            math.log(start_prob.get(state, 0) + epsilon) +
            math.log(emit_prob[state].get(first_obs, 0) + epsilon)
        )
        path[state] = [state]

    for t in range(1, len(obs_seq)):
        V.append({})
        new_path = {}
        current_obs = obs_seq[t]

        for curr_state in states:
            max_prob = float('-inf')
            best_prev_state = None

            for prev_state in states:
                prob = (
                    V[t - 1][prev_state] +
                    math.log(trans_prob[prev_state].get(curr_state, 0) + epsilon) +
                    math.log(emit_prob[curr_state].get(current_obs, 0) + epsilon)
                )
                if prob > max_prob:
                    max_prob = prob
                    best_prev_state = prev_state

            V[t][curr_state] = max_prob
            new_path[curr_state] = path[best_prev_state] + [curr_state]

        path = new_path


    final_state = max(V[-1], key=V[-1].get)
    return path[final_state]


observed_seq = "CTTCATGTGAAAGCAGACGTAAGTCA"
most_likely_path = viterbi(observed_seq, states, start_prob, trans_prob, emit_prob)
print("Most likely state path:", ''.join(most_likely_path))

Most likely state path: EEEEEEEEEEEEEEEEEEEEEEEEEE
